# Every GWAS mapped to an HP term, with a suggested disease term

98.2% of HP terms carry no therapeutic area (`../ta-distribution/MAPPING_REVIEW.md`), so every GWAS annotated
to one falls into the residual `other` bucket. This notebook lists them all and proposes, where one exists,
the EFO/MONDO/Orphanet disease term the study should have been mapped to instead — as a **curation worklist**,
not an applied change. Nothing downstream is modified.

**Scope** — the 1,394-term disease list behind gPS, i.e. diseases with at least one qualifying credible set
carrying an L2G-prioritised gene. Within it, **212 terms are HP**, used by **545 studies**, giving **560 rows**
(one per study × HP term).

## How the suggestion is produced, and why it cannot be invented

Two stages, deliberately separated:

1. **Retrieval — deterministic, from `disease.parquet` only.** For each HP term, a shortlist of up to 12
   candidate disease terms is built from the ontology by direct cross-reference, shared fine-grained
   cross-reference, and label/synonym token overlap, restricted to non-HP terms that map to a *real*
   therapeutic area (`other` and `sign or symptom` excluded as targets).
2. **Ranking — a small model picks from that shortlist or declines.** Eight Haiku agents each reviewed a
   chunk of the 212 terms. They were instructed that they may only return an id **present in that term's own
   candidate list**, and that declining is the expected answer for symptoms and generic abnormalities.

Every returned id is then re-validated against the ontology index *and* against the candidate list it was
drawn from. Any id failing either check is rejected and the row is downgraded — see the audit cell. **No
ontology identifier in this table comes from model memory.**

## Confidence scale (as specified)

| score | meaning |
| ----- | ------- |
| **1** | very confident an analogous disease term exists and the suggestion is correct |
| **2** | a plausible candidate exists but needs human checking |
| **NA** | no suitable disease term found |
| **0** | the GWAS-trait → HP-term mapping itself looks wrong |

In [1]:
import ast
import itertools
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

INTERMEDIATE = "../../../data/intermediate_files/"
RELEASE = "../../../data/25.06/"
VERDICTS = "hp_review_verdicts.json"  # frozen model output, versioned next to this notebook
CANDIDATES = "hp_review_candidates.json"  # frozen retrieval output, for the validation check

## Therapeutic-area map and the ontology

In [2]:
onto = pd.read_parquet(RELEASE + "output/disease/disease.parquet", columns=["id", "name", "description", "descendants"])
NAME = onto.set_index("id")["name"].to_dict()
DESCRIPTION = onto.set_index("id")["description"].to_dict()
INDEX = set(onto["id"])

THERAPY_AREA_HIERARCHY = {
    "MONDO_0045024": "cancer or benign tumor",
    "EFO_0005741": "infectious disease",
    "OTAR_0000009": "injury, poisoning or other complication",
    "OTAR_0000014": "pregnancy or perinatal disease",
    "MONDO_0024458": "disorder of visual system",
    "EFO_0000319": "cardiovascular disease",
    "EFO_0009605": "pancreas disease",
    "EFO_0000540": "immune system disease",
    "EFO_0010282": "gastrointestinal disease",
    "OTAR_0000017": "reproductive system or breast disease",
    "EFO_0010285": "integumentary system disease",
    "EFO_0001379": "endocrine system disease",
    "OTAR_0000010": "respiratory or thoracic disease",
    "EFO_0009690": "urinary system disease",
    "OTAR_0000006": "musculoskeletal or connective tissue disease",
    "MONDO_0021205": "disorder of ear",
    "EFO_0005803": "hematologic disease",
    "EFO_0000618": "nervous system disease",
    "MONDO_0002025": "psychiatric disorder",
    "OTAR_0000020": "nutritional or metabolic disease",
    "OTAR_0000018": "genetic, familial or congenital disease",
    "EFO_0003765": "sign or symptom",
}
DESC = {
    r.id: set(r.descendants) if r.descendants is not None else set()
    for r in onto[onto.id.isin(THERAPY_AREA_HIERARCHY)].itertuples()
}


def primary_area(term):
    for root, label in THERAPY_AREA_HIERARCHY.items():
        if term == root or term in DESC[root]:
            return label
    return "other"


print("ontology terms:", len(INDEX))

ontology terms: 38959


## The GWAS rows

One row per (study, HP term). Studies come from the same table the gPS disease list is built from, so the
universe matches the manuscript exactly.

In [3]:
genes_df = pd.read_csv(INTERMEDIATE + "genes_therapeutic_areas.csv")
l2g = pd.read_csv(INTERMEDIATE + "l2g_diseases_full-r1.csv", usecols=["studyId", "geneId", "diseaseIds"])
l2g = l2g[l2g["geneId"].isin(set(genes_df["geneId"]))]
studies = l2g.drop_duplicates("studyId").copy()
studies["terms"] = studies["diseaseIds"].map(ast.literal_eval)

gps_terms = set(itertools.chain.from_iterable(studies["terms"]))
assert len(gps_terms) == 1394, "the gPS disease list must be the published 1,394 terms"
HP_TERMS = sorted(t for t in gps_terms if t.startswith("HP_"))
print(f"gPS disease terms: {len(gps_terms)} | HP among them: {len(HP_TERMS)}")

study_index = pd.read_parquet(
    INTERMEDIATE + "qualifying_gwas_studies", columns=["studyId", "traitFromSource", "projectId", "nCases", "nSamples"]
)
meta = study_index.set_index("studyId")

rows = []
for r in studies.itertuples():
    for hp in [t for t in r.terms if t.startswith("HP_")]:
        rows.append(
            {
                "studyId": r.studyId,
                "projectId": meta.at[r.studyId, "projectId"] if r.studyId in meta.index else "",
                "traitFromSource": meta.at[r.studyId, "traitFromSource"] if r.studyId in meta.index else "",
                "all_diseaseIds": ";".join(r.terms),
                "n_diseaseIds": len(r.terms),
                "hp_term": hp,
                "hp_name": NAME.get(hp, ""),
                "hp_description": (DESCRIPTION.get(hp) or ""),
                "hp_current_therapeutic_area": primary_area(hp),
            }
        )
table = pd.DataFrame(rows)
print(
    f"rows (study x HP term): {len(table)} | studies: {table['studyId'].nunique()}"
    f" | HP terms: {table['hp_term'].nunique()}"
)
print()
print(table["hp_current_therapeutic_area"].value_counts().to_string())

gPS disease terms: 1394 | HP among them: 212
rows (study x HP term): 560 | studies: 545 | HP terms: 212

hp_current_therapeutic_area
other                        512
sign or symptom               41
cardiovascular disease         6
disorder of visual system      1


As expected, essentially every one of these studies currently sits in `other`.

## The suggested terms, and the anti-hallucination audit

`hp_review_verdicts.json` is the frozen output of the eight Haiku agents; `hp_review_candidates.json` is
the frozen retrieval shortlist each agent was allowed to choose from. Both are versioned next to this
notebook so the table is reproducible without re-running any model.

In [4]:
with open(VERDICTS) as f:
    verdicts = pd.DataFrame(json.load(f))
with open(CANDIDATES) as f:
    packet = json.load(f)
allowed = {p["hp_id"]: {c["id"] for c in p["candidates"]} for p in packet}

print(f"verdicts: {len(verdicts)} | HP terms expected: {len(HP_TERMS)}")
missing = set(HP_TERMS) - set(verdicts["hp_id"])
assert not missing, f"no verdict for {len(missing)} HP terms: {sorted(missing)[:5]}"
assert verdicts["hp_id"].is_unique, "duplicate verdicts"

suggested = verdicts["suggested_term"]
audit = pd.DataFrame(
    {
        "check": [
            "suggestion is non-null",
            "suggested id exists in disease.parquet",
            "suggested id was in that term's candidate list",
            "suggested id is not itself an HP term",
            "suggested id maps to a real therapeutic area",
        ],
        "n": [
            int(suggested.notna().sum()),
            int(sum(1 for s in suggested.dropna() if s in INDEX)),
            int(sum(1 for h, s in zip(verdicts["hp_id"], suggested) if s and s in allowed.get(h, set()))),
            int(sum(1 for s in suggested.dropna() if not str(s).startswith("HP_"))),
            int(sum(1 for s in suggested.dropna() if primary_area(s) not in ("other", "sign or symptom"))),
        ],
    }
)
print(audit.to_string(index=False))

bad = [(h, s) for h, s in zip(verdicts["hp_id"], suggested) if s and (s not in INDEX or s not in allowed.get(h, set()))]
print(f"\nrejected suggestions (invented or off-list): {len(bad)}")
for h, s in bad[:10]:
    print(f"   {h} -> {s}")

verdicts["rejected"] = [
    bool(s) and (s not in INDEX or s not in allowed.get(h, set())) for h, s in zip(verdicts["hp_id"], suggested)
]
verdicts.loc[verdicts["rejected"], "confidence"] = 2
verdicts.loc[verdicts["rejected"], "reason"] = (
    "suggestion rejected by the validation check (not in the ontology or not in the candidate list)"
)
verdicts.loc[verdicts["rejected"], "suggested_term"] = None

verdicts: 212 | HP terms expected: 212


                                         check  n
                        suggestion is non-null 96
        suggested id exists in disease.parquet 96
suggested id was in that term's candidate list 96
         suggested id is not itself an HP term 96
  suggested id maps to a real therapeutic area 96

rejected suggestions (invented or off-list): 0


### Correcting one misuse of the `0` code

`0` means *the GWAS trait does not match the HP term it was given*. Some verdicts instead used `0` to say
*the candidate list contained nothing relevant*, which is `NA`. The distinction is mechanical: a genuine `0`
has to be a statement about the trait, not about the candidates, so any `0` whose justification talks about
the candidate list is reclassified.

In [5]:
bad_zero = (verdicts["confidence"].astype(str) == "0") & verdicts["reason"].str.contains(
    "candidate", case=False, na=False
)
print("verdicts coded 0:", int((verdicts["confidence"].astype(str) == "0").sum()))
for r in verdicts[verdicts["confidence"].astype(str) == "0"].itertuples():
    tag = "-> reclassified NA" if bad_zero[r.Index] else "-> kept as 0"
    print(f"   {r.hp_id:14s} {NAME.get(r.hp_id, ''):32s} {tag}")
    print(f"      {r.reason}")

verdicts.loc[bad_zero, "confidence"] = "NA"
verdicts.loc[bad_zero, "reason"] = (
    verdicts.loc[bad_zero, "reason"] + " [recoded NA: complaint is about the candidate list, not the GWAS trait]"
)
print(
    f"\nreclassified {int(bad_zero.sum())} | genuine trait/HP mismatches remaining:"
    f" {int((verdicts['confidence'].astype(str) == '0').sum())}"
)
print()
print(verdicts["confidence"].astype(str).value_counts().to_string())

verdicts coded 0: 3
   HP_0000876     Oligomenorrhea                   -> kept as 0
      GWAS trait 'excessive, frequent and irregular menstruation' contradicts HP term 'oligomenorrhea' (infrequent); mapping is wrong.
   HP_0001943     Hypoglycemia                     -> reclassified NA
      GWAS mapping error: candidates are blood/vision/tumor diseases, none relate to hypoglycemia (low blood glucose).
   HP_0002105     Hemoptysis                       -> reclassified NA
      GWAS mapping error: candidates are blood disorders, none relate to hemoptysis (coughing up blood from respiratory tract).

reclassified 2 | genuine trait/HP mismatches remaining: 1

confidence
NA    115
1      70
2      26
0       1


## The final table

In [6]:
v = verdicts.set_index("hp_id")
table["suggested_term"] = table["hp_term"].map(v["suggested_term"])
table["suggested_name"] = table["suggested_term"].map(lambda t: NAME.get(t, "") if t else "")
table["suggested_therapeutic_area"] = table["suggested_term"].map(lambda t: primary_area(t) if t else "")
table["confidence"] = table["hp_term"].map(v["confidence"]).astype(str)
table["reason"] = table["hp_term"].map(v["reason"])
table["suggested_term_already_in_disease_list"] = table["suggested_term"].map(lambda t: bool(t) and t in gps_terms)

COLUMNS = [
    "studyId",
    "projectId",
    "traitFromSource",
    "all_diseaseIds",
    "n_diseaseIds",
    "hp_term",
    "hp_name",
    "hp_description",
    "hp_current_therapeutic_area",
    "suggested_term",
    "suggested_name",
    "suggested_therapeutic_area",
    "suggested_term_already_in_disease_list",
    "confidence",
    "reason",
]
table = table[COLUMNS].sort_values(["confidence", "hp_name", "studyId"]).reset_index(drop=True)
table.to_csv(INTERMEDIATE + "hp_mapped_gwas_review-r1.csv", index=False)
print(f"wrote hp_mapped_gwas_review-r1.csv  ({len(table)} rows, {len(COLUMNS)} columns)")
print()
print(table.head(8).to_string(index=False))

wrote hp_mapped_gwas_review-r1.csv  (560 rows, 15 columns)

                                 studyId   projectId                                           traitFromSource                   all_diseaseIds  n_diseaseIds    hp_term                            hp_name                                                                                                                                                        hp_description hp_current_therapeutic_area suggested_term             suggested_name       suggested_therapeutic_area  suggested_term_already_in_disease_list confidence                                                                                                                           reason
             FINNGEN_R12_N14_MESNRUIRREG FINNGEN_R12            Excessive, frequent and irregular menstruation HP_0000876;HP_0000132;HP_0400007             3 HP_0000876                     Oligomenorrhea                                                                                    

## Summary of the worklist

In [7]:
per_term = table.drop_duplicates("hp_term")
summary = (
    per_term.groupby("confidence")
    .agg(hp_terms=("hp_term", "size"))
    .join(table.groupby("confidence").agg(gwas_rows=("studyId", "size"), studies=("studyId", "nunique")))
    .reset_index()
)
labels = {
    "1": "1 - confident match",
    "2": "2 - needs checking",
    "NA": "NA - no disease term exists",
    "0": "0 - HP mapping itself looks wrong",
}
summary["meaning"] = summary["confidence"].map(labels)
summary.to_csv(INTERMEDIATE + "hp_mapped_gwas_review_summary-r1.csv", index=False)
print(summary.to_string(index=False))
print()
conf1 = per_term[per_term["confidence"] == "1"]
print(f"HP terms with a confident replacement: {len(conf1)} of {len(per_term)}")
print(
    f"  of which the target is ALREADY in the 1,394-disease list "
    f"(so merging removes a term): {int(conf1['suggested_term_already_in_disease_list'].sum())}"
)
print(f"  target is new to the list (relabel only): {int((~conf1['suggested_term_already_in_disease_list']).sum())}")
print()
print("therapeutic areas the confident matches would move into:")
print(conf1["suggested_therapeutic_area"].value_counts().to_string())

confidence  hp_terms  gwas_rows  studies                           meaning
         0         1          1        1 0 - HP mapping itself looks wrong
         1        70        232      231               1 - confident match
         2        26         60       60                2 - needs checking
        NA       115        267      256       NA - no disease term exists

HP terms with a confident replacement: 70 of 212
  of which the target is ALREADY in the 1,394-disease list (so merging removes a term): 31
  target is new to the list (relabel only): 39

therapeutic areas the confident matches would move into:
suggested_therapeutic_area
cardiovascular disease                          10
nervous system disease                           8
gastrointestinal disease                         8
disorder of visual system                        8
musculoskeletal or connective tissue disease     7
cancer or benign tumor                           5
immune system disease                         

In [8]:
print("--- confidence 1: apply these ---")
print(
    conf1.sort_values("hp_name")[
        ["hp_term", "hp_name", "suggested_term", "suggested_name", "suggested_therapeutic_area"]
    ].to_string(index=False)
)

--- confidence 1: apply these ---
   hp_term                                                       hp_name suggested_term                   suggested_name                   suggested_therapeutic_area
HP_0001892                                             Abnormal bleeding  MONDO_0002243              hemorrhagic disease                          hematologic disease
HP_0001671                            Abnormal cardiac septum morphology  MONDO_0002078              heart septal defect                       cardiovascular disease
HP_0011014                                  Abnormal glucose homeostasis    EFO_0009406       glucose metabolism disease             nutritional or metabolic disease
HP_0001977                                           Abnormal thrombosis  MONDO_0000831               thrombotic disease                       cardiovascular disease
HP_0100022                                       Abnormality of movement    EFO_0004280                movement disorder                

In [9]:
flagged = per_term[per_term["confidence"] == "0"]
print(f"--- confidence 0: possible GWAS-trait to HP-term mis-mapping ({len(flagged)} terms) ---")
if len(flagged):
    ex = (
        table[table["confidence"] == "0"]
        .groupby("hp_term")["traitFromSource"]
        .apply(lambda s: " | ".join(sorted(set(s))[:3]))
    )
    out = flagged[["hp_term", "hp_name", "reason"]].copy()
    out["example_gwas_traits"] = out["hp_term"].map(ex)
    print(out.to_string(index=False))

--- confidence 0: possible GWAS-trait to HP-term mis-mapping (1 terms) ---
   hp_term        hp_name                                                                                                                           reason                            example_gwas_traits
HP_0000876 Oligomenorrhea GWAS trait 'excessive, frequent and irregular menstruation' contradicts HP term 'oligomenorrhea' (infrequent); mapping is wrong. Excessive, frequent and irregular menstruation


In [10]:
na = per_term[per_term["confidence"] == "NA"]
counts = table[table["confidence"] == "NA"].groupby("hp_name")["studyId"].nunique().sort_values(ascending=False)
print(f"--- NA: no disease equivalent exists ({len(na)} terms, {int(counts.sum())} study rows) ---")
print("biggest by study count -- these are the terms the HP organ-system crosswalk has to handle instead:")
print(counts.head(25).to_string())

--- NA: no disease equivalent exists (115 terms, 267 study rows) ---
biggest by study count -- these are the terms the HP organ-system crosswalk has to handle instead:
hp_name
Inguinal hernia                          15
Iron deficiency anemia                   13
Back pain                                10
Hallux valgus                             7
Proteinuria                               7
Limb pain                                 6
Neck pain                                 6
Abnormality of limbs                      6
Syncope                                   6
Thrombophlebitis                          6
Umbilical hernia                          6
Abnormality of metabolism/homeostasis     5
Menorrhagia                               5
Urinary incontinence                      5
Sepsis                                    4
Moderate albuminuria                      4
Abdominal pain                            4
Hernia                                    4
Abnormality of the liver        

## TSV export for manual validation

Two sheets, tab-separated, with every field stripped of tabs and newlines so they open cleanly in Excel.
Both carry a `candidates_offered` column — the shortlist the model was restricted to — so a reviewer can see
not only what was picked but what else was available, and catch a case where a better option was on the list
and passed over.

- **`hp_mapped_gwas_review-r1.tsv`** — all 560 GWAS × HP rows (every one of the 70 + 26 + 115 + 1 HP terms,
  with all of its studies).
- **`hp_terms_review-r1.tsv`** — the same verdicts collapsed to **one row per HP term (212)**, with the study
  list in a single column. This is the sheet to actually validate: the 560-row version repeats each verdict
  once per study.

In [11]:
def clean(x):
    """TSV-safe: no tabs, no newlines, no doubled spaces."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return " ".join(str(x).replace("\t", " ").replace("\r", " ").replace("\n", " ").split())


offered = {p["hp_id"]: " | ".join(f"{c['id']}={c['name']}" for c in p["candidates"]) for p in packet}

wide = table.copy()
wide["candidates_offered"] = wide["hp_term"].map(offered)
wide["n_candidates_offered"] = wide["hp_term"].map(lambda h: len(allowed.get(h, ())))
wide["validated_ok"] = ""  # blank column for the reviewer to fill in
wide["reviewer_note"] = ""
for c in wide.columns:
    wide[c] = wide[c].map(clean)
order = [
    "confidence",
    "hp_term",
    "hp_name",
    "studyId",
    "projectId",
    "traitFromSource",
    "all_diseaseIds",
    "n_diseaseIds",
    "hp_description",
    "hp_current_therapeutic_area",
    "suggested_term",
    "suggested_name",
    "suggested_therapeutic_area",
    "suggested_term_already_in_disease_list",
    "reason",
    "n_candidates_offered",
    "candidates_offered",
    "validated_ok",
    "reviewer_note",
]
wide = wide[order].sort_values(["confidence", "hp_name", "studyId"])
wide.to_csv(INTERMEDIATE + "hp_mapped_gwas_review-r1.tsv", sep="\t", index=False)

studies_per_term = (
    table.groupby("hp_term")
    .apply(lambda d: " | ".join(f"{s} [{t}]" for s, t in zip(d["studyId"], d["traitFromSource"])), include_groups=False)
    .rename("gwas_studies")
)
# Build by assignment on a single indexed frame. Mixing Series (aligned on index) with bare
# arrays (placed positionally) in a DataFrame({...}) constructor silently misaligns the columns.
compact = (
    table.drop_duplicates("hp_term")
    .set_index("hp_term")[
        [
            "confidence",
            "hp_name",
            "hp_description",
            "hp_current_therapeutic_area",
            "suggested_term",
            "suggested_name",
            "suggested_therapeutic_area",
            "suggested_term_already_in_disease_list",
            "reason",
        ]
    ]
    .copy()
)
compact["n_gwas"] = table.groupby("hp_term")["studyId"].nunique()
compact["gwas_studies"] = studies_per_term
compact["n_candidates_offered"] = pd.Series({h: len(allowed.get(h, ())) for h in compact.index})
compact["candidates_offered"] = pd.Series({h: offered.get(h, "") for h in compact.index})
compact = compact.reset_index()
compact = compact[
    [
        "confidence",
        "hp_term",
        "hp_name",
        "hp_description",
        "n_gwas",
        "gwas_studies",
        "hp_current_therapeutic_area",
        "suggested_term",
        "suggested_name",
        "suggested_therapeutic_area",
        "suggested_term_already_in_disease_list",
        "reason",
        "n_candidates_offered",
        "candidates_offered",
    ]
]

# every hp_term must still carry its own name and verdict
check = compact.merge(
    table.drop_duplicates("hp_term")[["hp_term", "hp_name", "suggested_term"]], on="hp_term", suffixes=("", "_src")
)
assert (check["hp_name"] == check["hp_name_src"]).all(), "hp_name misaligned"
assert (check["suggested_term"].fillna("") == check["suggested_term_src"].fillna("")).all(), "suggested_term misaligned"

compact["validated_ok"] = ""
compact["reviewer_note"] = ""
for c in compact.columns:
    compact[c] = compact[c].map(clean)
compact = compact.sort_values(["confidence", "hp_name"])
compact.to_csv(INTERMEDIATE + "hp_terms_review-r1.tsv", sep="\t", index=False)

# a TSV with an embedded tab or newline would silently shift columns
for path, frame in [("hp_mapped_gwas_review-r1.tsv", wide), ("hp_terms_review-r1.tsv", compact)]:
    with open(INTERMEDIATE + path) as f:
        widths = {len(line.split("\t")) for line in f}
    assert len(widths) == 1, f"{path} has ragged rows: {widths}"
    print(f"{path:34s} {len(frame):4d} rows x {frame.shape[1]} cols, all rows {widths.pop()} fields")

print()
print("HP terms per confidence in the compact sheet:")
print(compact["confidence"].value_counts().to_string())
print(f"total HP terms: {len(compact)}  (70 + 26 + 115 + 1 = 212)")
assert len(compact) == 212 and len(wide) == 560

hp_mapped_gwas_review-r1.tsv        560 rows x 19 cols, all rows 19 fields
hp_terms_review-r1.tsv              212 rows x 16 cols, all rows 16 fields

HP terms per confidence in the compact sheet:
confidence
NA    115
1      70
2      26
0       1
total HP terms: 212  (70 + 26 + 115 + 1 = 212)


## What to do with this

- **Confidence 1** is the applyable list: a remap here removes a duplicate term where the target is already
  in the disease list, or simply relabels the study onto a term with a real therapeutic area.
- **Confidence 2** needs a human pass. The list is short enough to curate by hand.
- **NA** is the majority and is the important negative result: for most HP terms EFO/MONDO genuinely has no
  equivalent, because those ontologies model these as phenotypes by design. **Remapping cannot fix them —
  only an HP organ-system → therapeutic-area crosswalk can.**
- **Confidence 0** flags studies whose trait name does not match the HP term they were given; those are
  upstream curation bugs worth reporting rather than remapping.

## Limitations

- The suggestion is a model's choice from a deterministic shortlist. It is a **curation worklist, not an
  applied mapping**, and every confidence-1 row should still be eyeballed before use.
- Retrieval can only propose what token overlap or a cross-reference surfaces; a correct target with an
  unrelated label and no shared xref will be missed and shows up as NA.
- Frozen model output. Re-running the agents would not necessarily reproduce it verbatim; the validation
  cell is what guarantees the ids are real, not the model.

## Exports

| File | Contents |
| ---- | -------- |
| `hp_mapped_gwas_review-r1.csv` | one row per GWAS × HP term: study, trait, all disease ids, HP term + description, current area, suggested term + area, confidence, reason |
| `hp_mapped_gwas_review_summary-r1.csv` | counts of HP terms / studies / rows by confidence |
| `hp_review_verdicts.json`, `hp_review_candidates.json` | frozen model output and the shortlist it was restricted to, versioned in this folder |